> **English archival edition.** Saved outputs are historical observations, preserved rather than rerun. Chinese text has been translated. Use a new run label or output directory: translated source hashes differ from historical checkpoints.

# v7.3: Complete mechanism experiments after repairing normal tool execution

**Standalone notebook:** tasks, 96 new formal scenarios, 8 technical scenarios, frozen head coordinates, original tool code, and statistical analysis are embedded.

Retain the original Llama-3.1-8B BASE / Instruct revisions, BF16, and the actual `check_stock -> reserve_item -> create_order` state transitions. A public progress ledger and legal-action decoding prevent repeated idle queries and fabricated reservation identifiers. The model still chooses between correct and incorrect SKUs. Constrained-workflow completion is not unaided model capability.

First validate normal tool execution, then automatically run the complete Q/M/L/G design, the 80 candidate heads and random controls, final-order measurements, and matched free-decoding shadow controls. Stop and save evidence if acceptance checks fail. **Local software validation is complete; actual 8B acceptance is evaluated in this Colab run.**

Select an A100 80GB, enable notebook access to `HF_TOKEN` in Colab Secrets, and choose **Run all**. No previous runtime, new head search, or OpenAI API calls are required. The default soft budget is 22 hours; save checkpoints and release resources at the limit. Estimate duration from the pilot; completion within one 24-hour session is not guaranteed.


In [ ]:
#@title  Settings, Mounting Drive（ Keep running the same. RUN_LABEL）
import os,time,re,math
from pathlib import Path
from google.colab import drive
RUN_LABEL="main" #@param {type:"string"}
RUN_MODE="all" #@param ["all","pilot","export"]
MAX_WALL_HOURS=22.0 #@param {type:"number"}
AUTO_DISCONNECT=True #@param {type:"boolean"}
TAKEOVER_LOCK_ID="" #@param {type:"string"}
V73_STARTED=time.monotonic()
if not re.fullmatch(r"[A-Za-z0-9_-]{1,60}",RUN_LABEL):raise ValueError("RUN_LABEL Only letters, numbers, underlinings and short crosslines may be used. ")
if not math.isfinite(MAX_WALL_HOURS) or not 0<MAX_WALL_HOURS<=23:raise ValueError(" The budget must be larger than 0 and not more than 23 Hours. ")
DRIVE_MOUNT=Path("/content/drive")
drive.mount(str(DRIVE_MOUNT))
if not os.path.ismount(str(DRIVE_MOUNT)) or not (DRIVE_MOUNT/"MyDrive").is_dir():raise RuntimeError(" Please finish. Drive Mount. ")
V73_RUN=DRIVE_MOUNT/"MyDrive/AAAI_Workshop/agent_entrainment/agent_entrainment_v7p3_execution_mechanism_confirm96"/("execution-"+RUN_LABEL)
print(" Results catalogue: ",V73_RUN)


In [ ]:
if RUN_MODE != "export":
    import sys, subprocess, importlib.metadata as v5_metadata
    if not ((3, 10) <= sys.version_info[:2] <= (3, 13)):
        raise RuntimeError(' Fixed-dependent combination needs  Python 3.10–3.13。')
    V6_REQUIRED_VERSIONS = {
        'transformer-lens': '2.17.0', 'transformers': '4.57.1', 'huggingface-hub': '0.36.0',
        'numpy': '2.1.3' if sys.version_info[:2] == (3, 13) else '1.26.4',
        'pandas': '2.2.3', 'plotly': '6.1.2', 'scipy': '1.15.3',
        'scikit-learn': '1.6.1', 'datasets': '3.6.0', 'pyarrow': '20.0.0',
    }
    V6_INSTALL_PROBLEMS = []
    for v5_name, v5_version in V6_REQUIRED_VERSIONS.items():
        try:
            v5_installed = v5_metadata.version(v5_name)
        except v5_metadata.PackageNotFoundError:
            v5_installed = None
        if v5_installed != v5_version:
            V6_INSTALL_PROBLEMS.append((v5_name, v5_installed, v5_version))
    if V6_INSTALL_PROBLEMS:
        print(' The original experimental fixed version will be installed: ', V6_INSTALL_PROBLEMS)
        subprocess.run([sys.executable, '-m', 'pip', 'install',
                        *[f'{name}=={version}' for name, version in V6_REQUIRED_VERSIONS.items()], 'tqdm'], check=True)
        raise SystemExit(' Installation complete. Please restart  Colab runtime， Again.  Run all； Versions are not repeated after matching. ')
    print(' Fixed reliance matches, skips installation. ')

else:
    print(" Read-only export: Skip this step. ")


In [ ]:
if RUN_MODE != "export":
    #  Run this block after restarting: check if the installation is currently in place  Python  Environment. 
    import sys
    import importlib.metadata as metadata
    import importlib.util

    print("Python:", sys.version.split()[0])
    print(" Interpreter :", sys.executable)
    if not ((3, 10) <= sys.version_info[:2] <= (3, 13)):
        raise RuntimeError(" Yes.  Python 3.10–3.13； Please re-implement the installment block after running. ")

    expected_versions = {
        "transformer-lens": "2.17.0", "transformers": "4.57.1",
        "huggingface-hub": "0.36.0",
        "numpy": "2.1.3" if sys.version_info[:2] == (3, 13) else "1.26.4",
        "pandas": "2.2.3",
        "plotly": "6.1.2", "scipy": "1.15.3", "scikit-learn": "1.6.1",
        "datasets": "3.6.0", "pyarrow": "20.0.0",
    }
    problems = []
    for package, expected in expected_versions.items():
        try:
            installed = metadata.version(package)
        except metadata.PackageNotFoundError:
            installed = None
        print(f"{package}: {installed if installed is not None else ' not installed '} ( Yes.  {expected})")
        if installed != expected:
            problems.append(f"{package}:  current  {installed}， Yes.  {expected}")

    for package in ("torch", "tqdm"):
        try:
            print(f"{package}: {metadata.version(package)}")
        except metadata.PackageNotFoundError:
            problems.append(f"{package}:  not installed ")

    try:
        torch_version = metadata.version("torch")
        torch_major_minor = tuple(int(part) for part in torch_version.split("+")[0].split(".")[:2])
        if torch_major_minor < (2, 6):
            problems.append(f"torch:  current  {torch_version}， Yes.  >=2.6")
    except metadata.PackageNotFoundError:
        pass  #  The record is missing. 
    except ValueError:
        problems.append(" Cannot parse  torch  Version, please check the current version  Colab  Run when. ")

    if importlib.util.find_spec("transformer_lens") is None:
        problems.append(" current  Python  Cannot find  transformer_lens  module ")
    if problems:
        raise RuntimeError(
            " Reliance check failed: \n- " + "\n- ".join(problems)
            + "\n Run the first cell and inspect its complete installation log. After successful installation, restart the runtime, "
              " Go on from this block. "
        )
    print(" Reliance on a version check pass; continues to run the third code block. ")

else:
    print(" Read-only export: Skip this step. ")


In [ ]:
# Authenticate before accessing gated Hugging Face model files.
from google.colab import userdata
from huggingface_hub import login
try:
    _archive_hf_token = userdata.get("HF_TOKEN")
except Exception:
    raise RuntimeError("Set HF_TOKEN in Colab Secrets and enable notebook access.") from None
if not isinstance(_archive_hf_token, str) or not _archive_hf_token.strip():
    raise RuntimeError("HF_TOKEN is empty. Set it in Colab Secrets and enable notebook access.")
login(token=_archive_hf_token.strip(), add_to_git_credential=False)
del _archive_hf_token


In [ ]:
if RUN_MODE != "export":
    from google.colab import userdata
    from huggingface_hub import login
    def authenticate_hub():
        hint = " Please.  Colab Secrets  Set Readable  Llama  Model.  HF_TOKEN， ♪ And open this book ♪  notebook  Access rights. "
        try:
            token = userdata.get("HF_TOKEN")
        except Exception:
            raise RuntimeError(hint) from None
        if not isinstance(token,str) or not token.strip():
            raise RuntimeError(hint)
        try:
            login(token=token.strip(), add_to_git_credential=False)
        except Exception:
            raise RuntimeError(hint) from None
    authenticate_hub()

else:
    print(" Read-only export: Skip this step. ")


In [ ]:
#@title  Import run library (%1) export No model, no access. Hub）
import base64,hashlib,json,zlib,importlib.metadata
V73_DEPS={};V73_PACKAGES={}
if RUN_MODE!="export":
    os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF","expandable_segments:True")
    import numpy as np
    import torch
    from transformer_lens import HookedTransformer
    from huggingface_hub import get_token
    def disabled_login73():raise RuntimeError(" Please set Secrets HF_TOKEN ♪ And open ♪ notebook Accesses, reruns the login grid. ")
    V73_DEPS=dict(np=np,torch=torch,HookedTransformer=HookedTransformer,get_token=get_token,notebook_login=disabled_login73)
    V73_NAMES=("torch","transformer-lens","transformers","huggingface-hub","numpy","pandas","datasets","pyarrow","scipy","scikit-learn","plotly")
    V73_PACKAGES={k:importlib.metadata.version(k) for k in V73_NAMES}


In [ ]:
#@title  Unpackage and verify complete frozen source, task and statistics 
V73_ASSETS={k:base64.b64decode(v) for k,v in json.loads(zlib.decompress(base64.b64decode('...'))).items()}
V73_HASHES={'code/bootstrap73.py': '0660583369e852c7fe22dc6895a025b3af0dd7e6e4955aadbb7ac53ad56ebc7a', 'code/engine73.py': '94f1a52f707c65d3537aea42e4e0aab2e9c73173f251d86a277e258531224beb', 'code/frozen/accelerator71.py': '1a3cbe6156d2def298373be354c949c75e49a6dd853a0e74b3897b3ad7fe3468', 'code/frozen/reports70.py': '2ba21a3c7a6cbb783c585626b5a28c34b01693f02cdd61a03846a4702cc5ff7c', 'code/frozen/runtime72.py': '26e7640eb133018eed45ba64cd93d91acd75d38ad4395127f78408087f462375', 'code/frozen/study70.py': '5bbcd873b325471e08fccc83bad5943ca5a51a3ecc8c179160f5e3515e548721', 'code/frozen/v61_aligned_source.py': '39fec76288d585eeb7ed01f35b61bfe8f2f51d3bff24f8fddbaa0266ff1c31bb', 'code/frozen/v6_core_source.py': 'dfee0014eb1746e959d1b652d152093f00ea90782932900bd2123ca5afee7c53', 'code/frozen/v6_legacy_source.py': 'ecc77fb8592acfda16f19342d276672567fff0c65fec292f92abec4d4b5bf1af', 'code/frozen/v6_runtime_source.py': '02759ddf3a5c525a0f22d558ff2b4aedf25f1afc3fa09987a3f8bc2ec51b1bb7', 'code/frozen/v6_tasks_source.py': '4e9f82772209d475ea35d57a9dc6c4044281a63bc07c47a574848972e0212f59', 'code/frozen/v70_engine70_source.py': 'a9bd44bcfdac1e3fd5e4efe90cc72bd479c7d000033467dffa52bb364cee1bbc', 'code/frozen/v70_tasks70_source.py': 'f26b83c06043f2b3c4d08d601aac9b7fd01cce6a4212d57965b9e50083e236cf', 'code/frozen/v71_model_identity.json': 'ba52ac8f52b96d3a5c082289d1bc8fa1fbefbfa7735ad495ce9be842eb021c01', 'code/protocol73.py': 'f67880e35475e81c8bd7758745d43fcde46df5a0a750062d3bde54bfb975438d', 'code/runtime73.py': '0a39259d7011427c809da8bcdd59878ee5081b6d0fbdd3adf1518551a5e009f5', 'code/study73.py': '7d57765b9872df571e810a2b07aa7a09bf020ad15f493846dacc835a8e187a33', 'inputs/allocation.json': '479fc81a8e488fc7f3cf81020077ca1a39778fe11160e88bad4a8af1be548341', 'inputs/arms.json': 'fa25578b34c36c115994a55aa7d2e9204b0630e2227db43d1c1fd19e1d374e62', 'inputs/auxiliary-cells.json': '50001b2b7b54a53f2e5ab0119b2bf841a79e570ea67388f3922b885c8aeac695', 'inputs/cells.json': 'd05cd7844dd5cbf0fce162de7e4c02e753c1755e6f47ac62c2580f8b6fcf715f', 'inputs/collision_sources.json': '7750126ca71fc7f46aaa5cc66b38cb11de94574addd1303e8630f78b96976431', 'inputs/design.json': '9ec982d0f4f6c7817c2af5528ec0c7176c70ffca7bee4b612626cfe45f7e6421', 'inputs/example-donor-Q.txt': '288e1b4b6fa2c90c285c0b8ad6d44f1a4ae5de734e14bac09515019ea4c015b2', 'inputs/example-recipient.txt': '27c7d9f5b0b1c6b15d68409d9aa818218dccf9c04b72588c46f07ee63932caff', 'inputs/forbidden_identifiers.json': 'f739338fb958dddb3bf4372de3f81d51a9b39018f8251e573f5580beeec798ec', 'inputs/protocol.md': '415af0b6233faf40a98499e1a99325675d5c499dea14eb08b8b0ed224cc5b05b', 'inputs/setup-cells.json': '2725ee0c0021db616f60f5b2f4fbff14bc6c0e8862312223b693363f9725c578', 'inputs/worlds.json': '47992c2d32696164a3c950a60e42c00471ff2a5f99ce56575987c64d9415564a'}
if {k:hashlib.sha256(v).hexdigest() for k,v in V73_ASSETS.items()}!=V73_HASHES:raise RuntimeError(" Inline asset summary changes. ")
V73_BOOT={}
exec(compile(V73_ASSETS["code/bootstrap73.py"].decode(),"bootstrap73.py","exec"),V73_BOOT)
V73_NS=V73_BOOT["load_namespace73"](V73_ASSETS,V73_DEPS)
print(json.loads(V73_ASSETS["inputs/design.json"])["planned_counts"])


## Interpreting execution repair and mechanism results

- Each model first completes 32 normal tasks: empty history, stock already checked, an existing reservation, and an out-of-stock requested item. Proceed only with at least 30 successes, at least 7/8 at each starting state, and zero format or tool failures.
- The full main experiment contains 65,952 raw logit probes and 8,448 constrained trajectories. Together with 64 acceptance trajectories and 640 matched-prompt free-decoding controls, this gives 75,104 units.
- The 96 formal scenarios are the independent statistical units. Probes, templates, and head sets are paired conditions, not extra samples. Both primary metrics compare candidates against 19 random sets: query-minus-catalog transfer and label transfer.
- `item` precedes `tool`. Patching occurs once, before the first distinguishing SKU digit token, so the subsequent tool type may change. Save raw logits, raw argmax, and the actual constrained token separately.
- Every incorrect SKU remains selectable, reservable, and orderable. Report purchases of X, Y, fallback items, other items, and no orders together. Free-decoding shadow controls quantify executor assistance.


In [ ]:
#@title  Action. / Continue full experiment (acceptance failed to save automatically) PARTIAL）
V73_RESULT=V73_SESSION=V73_STUDY=None
with V73_NS["writer70"](V73_RUN,TAKEOVER_LOCK_ID):
    V73_IDENTITY=V73_NS["identity73"](V73_RUN,V73_ASSETS,V73_PACKAGES,export=RUN_MODE=="export")
    V73_STUDY=V73_NS["Study73"](V73_RUN,V73_ASSETS,V73_IDENTITY,export=RUN_MODE=="export")
    try:
        V73_SESSION=V73_NS["run73"](V73_STUDY,RUN_MODE,V73_STARTED,MAX_WALL_HOURS)
    except (Exception,KeyboardInterrupt) as error:
        V73_SESSION={"status":"ERROR","error_type":type(error).__name__}
        print(" Execution stopped; completed units retained Drive。 Runable under-barrel export PARTIAL； Hold RUN_LABEL Continue. ",flush=True)
        raise
print(" Session Status: ",V73_SESSION["status"])


In [ ]:
#@title  Export true results ZIP： Replay tool, recalculate report, check CRC/ List / Signature 
if V73_STUDY is None:raise RuntimeError(" Runs the upper grid first; export only possible RUN_MODE='export' Again. Run all。")
with V73_NS["writer70"](V73_RUN):
    V73_RESULT=V73_NS["export73"](V73_STUDY,V73_SESSION)
print(" Results: ",V73_RESULT["report"]["status"])
print(" Acceptance and inspection: ",V73_RESULT["report"]["readiness"])
print("ZIP：",V73_RESULT["path"])
print(" Completeness: ",V73_RESULT["audit"])


## Checkpoints, running costs, and evidence

Rerun **Run all** with the same `RUN_LABEL` to validate and skip saved units. Restore the original Python, dependency, and GPU identities; never mix changed code or another model into an existing run label. Take over a write lock only after confirming that the old runtime has stopped and supplying its exact identifier.

At the budget limit, save `PARTIAL` and later resume under the same label. `pilot` runs acceptance and technical tests only; switch back to `all` afterward for the formal stage. `export` needs neither a GPU nor model login and only recalculates reports and the ZIP. Failed normal-execution acceptance stops execution without automatically changing prompts or selecting scenarios.

`COMPLETE` requires all main and auxiliary controls. The report scope is `v7p3_execution_mechanism_confirm96`. The archive contains complete prompts, per-token constraint audits, donor/recipient scores, genuine tool logs, and final orders. No Writer rewrites the task, and historical-version results remain unchanged.


In [ ]:
#@title ZIP Deposit Drive And automatically disconnect when synchronised is successful runtime
from google.colab import drive,runtime
V73_NS["finish73"](V73_STUDY,V73_RESULT,V73_SESSION,AUTO_DISCONNECT and RUN_MODE!="export",
    DRIVE_MOUNT,os.path.ismount,drive.flush_and_unmount,runtime.unassign)
